In [ ]:
# product_id, url, review_text, user_info

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import ElementClickInterceptedException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from deep_translator import GoogleTranslator
import pandas as pd, time, traceback

# =====================================================
# ⚙️ 설정
# =====================================================
RANKING_URL = (
    "https://www.musinsa.com/main/musinsa/ranking?"
    "gf=A&storeCode=musinsa&sectionId=199&contentsId=&"
    "categoryCode=103000&ageBand=AGE_BAND_ALL&subPan=product&period=MONTHLY"
)
TARGET_URLS = 1000
REVIEW_LIMIT = 5
SAVE_INTERVAL = 50

# CSS Selector
SNAP_BTN = (
    "#root > div.Layout__Container-sc-3weaze-0.cbLSDw > "
    "div.VariableArea__Container-sc-4n9q35-0.bliLNC > "
    "div.ContentsTab__Container-sc-g3hx4t-0.jYxqLx > div > "
    "button.ContentsTab__Button-sc-g3hx4t-2.bYBIQW.gtm-click-button"
)
REVIEW_BLOCK = "#productDetailReviewSection div.sc-1pbmvpl-0.grtmxQ"
REVIEW_TEXT = "div.sc-1kxmu5p-0.jEvCTQ.gtm-click-button"
USER_INFO_SELECTOR = (
    "#productDetailReviewSection div.sc-etuca1-0.dvHbOb "
    "div > div:nth-child(2) > span.text-body_13px_reg.text-black.font-pretendard"
)

# product_id CSS 예외 케이스 두 개 모두 검사
PRODUCT_ID_CANDIDATES = [
    "#root > div.Layout__Container-sc-3weaze-0.cbLSDw > div.VariableArea__Container-sc-4n9q35-0.bliLNC > "
    "div:nth-child(4) > div.ContentsLayout__Wrap-sc-1bn3xag-0.hZXFPB > "
    "div.ContentsLayout__Inner-sc-1bn3xag-1.exlEdD > div > dl > div:nth-child(1) > dd",
    "#root > div.Layout__Container-sc-3weaze-0.cbLSDw > div.VariableArea__Container-sc-4n9q35-0.bliLNC > "
    "div:nth-child(5) > div.ContentsLayout__Wrap-sc-1bn3xag-0.hZXFPB > "
    "div.ContentsLayout__Inner-sc-1bn3xag-1.exlEdD > div > dl > div:nth-child(1) > dd"
]

# =====================================================
# 🌐 드라이버 설정
# =====================================================
options = Options()
options.add_argument("--window-size=1400,900")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
# options.add_argument("--headless")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 15)

# =====================================================
# 🌍 번역 함수
# =====================================================
translator = GoogleTranslator(source="ko", target="en")

def translate_auto(text):
    """한글 → 영어 자동 번역 (deep_translator)"""
    if not text:
        return None
    try:
        return translator.translate(text)
    except:
        return text  # 번역 실패 시 원문 유지

# =====================================================
# 🧩 헬퍼 함수
# =====================================================
def safe_get_product_id():
    """상품 ID를 여러 CSS 시도로 추출"""
    for sel in PRODUCT_ID_CANDIDATES:
        try:
            pid = driver.find_element(By.CSS_SELECTOR, sel).text.strip()
            if pid:
                return pid
        except:
            continue
    return None

def click_snap_tab():
    """후기 탭 클릭 (오버레이 감지 및 재시도)"""
    for attempt in range(3):
        try:
            btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, SNAP_BTN)))
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            time.sleep(0.8)

            # 오버레이 제거 대기
            try:
                WebDriverWait(driver, 5).until_not(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.Dimmed-sc-53e7ju-0"))
                )
            except TimeoutException:
                pass

            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2.5)
            print("✅ 후기 탭 클릭 성공")
            return True
        except Exception as e:
            print(f"⚠️ 클릭 재시도 ({attempt+1}/3): {e}")
            time.sleep(1.2)
    return False

def scroll_until_reviews(limit=REVIEW_LIMIT):
    for _ in range(8):
        driver.execute_script("window.scrollBy(0, window.innerHeight * 0.9);")
        time.sleep(1.0)
        if len(driver.find_elements(By.CSS_SELECTOR, REVIEW_BLOCK)) >= limit:
            break

# =====================================================
# 🔍 랭킹 페이지 URL 수집
# =====================================================
driver.get(RANKING_URL)
time.sleep(7)

collected = set()
last_count = 0
no_gain = 0
for step in range(120):
    driver.execute_script("window.scrollBy(0, window.innerHeight * 0.9);")
    time.sleep(0.8)
    hrefs = driver.execute_script("""
        return Array.from(document.querySelectorAll('a[href*="/products/"]'))
                     .map(a => a.href.split('?')[0]);
    """)
    collected.update([h for h in hrefs if "/products/" in h])
    gain = len(collected) - last_count
    last_count = len(collected)
    print(f"📦 스크롤 {step+1:03d} ▶ 누적 {len(collected)}개 (+{gain})")
    if gain == 0:
        no_gain += 1
    else:
        no_gain = 0
    if no_gain >= 6 or len(collected) >= TARGET_URLS:
        print("✅ 스크롤 완료 조건 충족")
        break

urls = list(collected)[:TARGET_URLS]
print(f"🎯 최종 상품 URL 수집 완료: {len(urls)}개")

# =====================================================
# 💬 후기 수집
# =====================================================
data, errors = [], []
for idx, url in enumerate(urls, start=1):
    print(f"\n▶ {idx}/{len(urls)} | {url}")
    try:
        driver.get(url)
        time.sleep(3.5)

        product_id = safe_get_product_id() or f"PID_FALLBACK_{idx:03d}"

        tab_clicked = click_snap_tab()
        if tab_clicked:
            scroll_until_reviews(REVIEW_LIMIT)
            time.sleep(1.5)

        review_blocks = driver.find_elements(By.CSS_SELECTOR, REVIEW_BLOCK)[:REVIEW_LIMIT]
        review_texts, user_infos = [], []

        for b in review_blocks:
            try:
                review_texts.append(b.find_element(By.CSS_SELECTOR, REVIEW_TEXT).text.strip())
            except:
                review_texts.append(None)

        user_infos_all = driver.find_elements(By.CSS_SELECTOR, USER_INFO_SELECTOR)
        for u in user_infos_all[:len(review_blocks)]:
            raw_text = u.text.strip()
            translated = translate_auto(raw_text)
            user_infos.append(translated)

        # 후기 존재 시 저장
        if review_texts:
            for i in range(len(review_texts)):
                data.append({
                    "product_id": product_id,
                    "url": url,
                    "review_text": review_texts[i],
                    "user_info_en": user_infos[i] if i < len(user_infos) else None
                })
        else:
            data.append({
                "product_id": product_id,
                "url": url,
                "review_text": None,
                "user_info_en": None
            })

        print(f"💬 후기 {len(review_texts)}개 수집 완료")

        if idx % SAVE_INTERVAL == 0:
            temp_file = f"musinsa_reviews_shoes_{idx}.xlsx"
            pd.DataFrame(data).to_excel(temp_file, index=False)
            print(f"💾 {idx}개 상품 중간 저장 완료 ({temp_file})")

    except Exception as e:
        print(f"⚠️ 오류 발생: {url}")
        errors.append({"url": url, "error": str(e), "traceback": traceback.format_exc()})
        continue

# =====================================================
# 💾 최종 저장
# =====================================================
driver.quit()
pd.DataFrame(data).to_excel("C04_1000_shoes.xlsx", index=False)
pd.DataFrame(errors).to_excel("C04_1000shoes_log.xlsx", index=False)

print("\n🎯 수집 완료")
print("⚙️ 오류 로그")

📦 스크롤 001 ▶ 누적 23개 (+23)
📦 스크롤 002 ▶ 누적 41개 (+18)
📦 스크롤 003 ▶ 누적 59개 (+18)
📦 스크롤 004 ▶ 누적 59개 (+0)
📦 스크롤 005 ▶ 누적 77개 (+18)
📦 스크롤 006 ▶ 누적 77개 (+0)
📦 스크롤 007 ▶ 누적 95개 (+18)
📦 스크롤 008 ▶ 누적 95개 (+0)
📦 스크롤 009 ▶ 누적 113개 (+18)
📦 스크롤 010 ▶ 누적 113개 (+0)
📦 스크롤 011 ▶ 누적 131개 (+18)
📦 스크롤 012 ▶ 누적 131개 (+0)
📦 스크롤 013 ▶ 누적 149개 (+18)
📦 스크롤 014 ▶ 누적 167개 (+18)
📦 스크롤 015 ▶ 누적 167개 (+0)
📦 스크롤 016 ▶ 누적 185개 (+18)
📦 스크롤 017 ▶ 누적 185개 (+0)
📦 스크롤 018 ▶ 누적 203개 (+18)
📦 스크롤 019 ▶ 누적 203개 (+0)
📦 스크롤 020 ▶ 누적 221개 (+18)
📦 스크롤 021 ▶ 누적 221개 (+0)
📦 스크롤 022 ▶ 누적 239개 (+18)
📦 스크롤 023 ▶ 누적 257개 (+18)
📦 스크롤 024 ▶ 누적 257개 (+0)
📦 스크롤 025 ▶ 누적 275개 (+18)
📦 스크롤 026 ▶ 누적 275개 (+0)
📦 스크롤 027 ▶ 누적 293개 (+18)
📦 스크롤 028 ▶ 누적 293개 (+0)
📦 스크롤 029 ▶ 누적 311개 (+18)
📦 스크롤 030 ▶ 누적 311개 (+0)
📦 스크롤 031 ▶ 누적 329개 (+18)
📦 스크롤 032 ▶ 누적 329개 (+0)
📦 스크롤 033 ▶ 누적 347개 (+18)
📦 스크롤 034 ▶ 누적 365개 (+18)
📦 스크롤 035 ▶ 누적 365개 (+0)
📦 스크롤 036 ▶ 누적 383개 (+18)
📦 스크롤 037 ▶ 누적 383개 (+0)
📦 스크롤 038 ▶ 누적 401개 (+18)
📦 스크롤 039 ▶ 누적 401개 (+0)
📦 스크롤 040 ▶